In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
import xgboost as xgb
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [2]:
!pip install xgboost lightgbm

In [3]:
df = pd.read_csv("covtype.csv")
df

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type32,Soil_Type33,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Cover_Type
0,2596,51,3,258,0,510,221,232,148,6279,...,0,0,0,0,0,0,0,0,0,5
1,2590,56,2,212,-6,390,220,235,151,6225,...,0,0,0,0,0,0,0,0,0,5
2,2804,139,9,268,65,3180,234,238,135,6121,...,0,0,0,0,0,0,0,0,0,2
3,2785,155,18,242,118,3090,238,238,122,6211,...,0,0,0,0,0,0,0,0,0,2
4,2595,45,2,153,-1,391,220,234,150,6172,...,0,0,0,0,0,0,0,0,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581007,2396,153,20,85,17,108,240,237,118,837,...,0,0,0,0,0,0,0,0,0,3
581008,2391,152,19,67,12,95,240,237,119,845,...,0,0,0,0,0,0,0,0,0,3
581009,2386,159,17,60,7,90,236,241,130,854,...,0,0,0,0,0,0,0,0,0,3
581010,2384,170,15,60,5,90,230,245,143,864,...,0,0,0,0,0,0,0,0,0,3


In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df.iloc[:, -1] = le.fit_transform(df.iloc[:, -1]) 

In [6]:
y = df.iloc[:, -1]  
X = df.iloc[:, :-1]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
rf_model = RandomForestClassifier(n_estimators=10, random_state=42)

In [10]:
rf_model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=10, random_state=42)

In [15]:
y_pred_rf = rf_model.predict(X_test)

In [11]:
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')

In [12]:
xgb_model.fit(X_train, y_train)

C:\Users\nensi\anaconda3\lib\site-packages\xgboost\core.py:158: UserWarning: [12:48:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, objective='multi:softprob', ...)

In [16]:
y_pred_xgb = xgb_model.predict(X_test)

In [13]:
lgbm_model = LGBMClassifier()

In [14]:
lgbm_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.037740 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2148
[LightGBM] [Info] Number of data points in the train set: 464809, number of used features: 53
[LightGBM] [Info] Start training from score -1.010055
[LightGBM] [Info] Start training from score -0.717554
[LightGBM] [Info] Start training from score -2.787067
[LightGBM] [Info] Start training from score -5.343669
[LightGBM] [Info] Start training from score -4.126990
[LightGBM] [Info] Start training from score -3.511322
[LightGBM] [Info] Start training from score -3.338569


LGBMClassifier()

In [17]:
y_pred_lgbm = lgbm_model.predict(X_test)

In [18]:
ensemble_model = VotingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model), ('lgbm', lgbm_model)], voting='hard')
ensemble_model.fit(X_train, y_train)
y_pred_ensemble = ensemble_model.predict(X_test)

C:\Users\nensi\anaconda3\lib\site-packages\xgboost\core.py:158: UserWarning: [12:50:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.041563 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2148
[LightGBM] [Info] Number of data points in the train set: 464809, number of used features: 53
[LightGBM] [Info] Start training from score -1.010055
[LightGBM] [Info] Start training from score -0.717554
[LightGBM] [Info] Start training from score -2.787067
[LightGBM] [Info] Start training from score -5.343669
[LightGBM] [Info] Start training from score -4.126990
[LightGBM] [Info] Start training from score -3.511322
[LightGBM] [Info] Start training from score -3.338569


In [19]:
nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(len(y.unique()), activation='softmax')
])

C:\Users\nensi\anaconda3\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
nn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
nn_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
11621/11621 ━━━━━━━━━━━━━━━━━━━━ 30s 2ms/step - accuracy: 0.6971 - loss: 0.7177 - val_accuracy: 0.7756 - val_loss: 0.5188
Epoch 2/10
11621/11621 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - accuracy: 0.7591 - loss: 0.5616 - val_accuracy: 0.7947 - val_loss: 0.4786
Epoch 3/10
11621/11621 ━━━━━━━━━━━━━━━━━━━━ 31s 3ms/step - accuracy: 0.7737 - loss: 0.5292 - val_accuracy: 0.8070 - val_loss: 0.4557
Epoch 4/10
11621/11621 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - accuracy: 0.7837 - loss: 0.5115 - val_accuracy: 0.8157 - val_loss: 0.4401
Epoch 5/10
11621/11621 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - accuracy: 0.7881 - loss: 0.5007 - val_accuracy: 0.8212 - val_loss: 0.4322
Epoch 6/10
11621/11621 ━━━━━━━━━━━━━━━━━━━━ 29s 3ms/step - accuracy: 0.7927 - loss: 0.4903 - val_accuracy: 0.8279 - val_loss: 0.4181
Epoch 7/10
11621/11621 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - accuracy: 0.7957 - loss: 0.4841 - val_accuracy: 0.8290 - val_loss: 0.4096
Epoch 8/10
11621/11621 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - accuracy: 

In [21]:
nn_pred = np.argmax(nn_model.predict(X_test), axis=1)

3632/3632 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step


In [22]:
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("LightGBM Accuracy:", accuracy_score(y_test, y_pred_lgbm))
print("Ensemble Model Accuracy:", accuracy_score(y_test, y_pred_ensemble))
print("Neural Network Accuracy:", accuracy_score(y_test, nn_pred))

Random Forest Accuracy: 0.9442871526552671
XGBoost Accuracy: 0.8711823274786366
LightGBM Accuracy: 0.8485839436159135
Ensemble Model Accuracy: 0.8842542791494196
Neural Network Accuracy: 0.835752949579615


In [23]:
from sklearn.decomposition import PCA
pca = PCA()
X_pca = pca.fit_transform(X_train)

In [24]:
explained_variance = pca.explained_variance_ratio_.cumsum()
optimal_components = next(i for i, v in enumerate(explained_variance) if v >= 0.95) + 1
print(f"Optimal number of PCA components: {optimal_components}")

Optimal number of PCA components: 43
